# 12 — From States to the Density Matrix

Promote a state vector to an operator — the density matrix $\rho = |\psi\rangle\langle\psi|$ — build it by hand, read where each piece of information lives in it, redo everything you know (Born rule, expectation values, gates) in its language, and meet the central character of the rest of the course.

**Prerequisites:** `01/03_the_bloch_sphere` (the $(\theta, \phi)$ parametrisation), `01/04_measurement_and_born_rule`, `01/05_inner_product_and_dirac` (bras, kets, $\langle\varphi|\psi\rangle$), `01/06_gates_as_unitary_evolution` (the bra of $U|\psi\rangle$), `01/07_pauli_gates` (the Paulis, the bracket $\langle\psi|A|\psi\rangle$), `01/10_expectation_values` (expectation values; the formula $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ you were asked to accept as a preview), `01/11_shot_noise_and_first_circuit` (the slot a vector does not have).

**What you'll be able to do**
- Build $\rho = |\psi\rangle\langle\psi|$ by hand from two amplitudes, and see a global phase disappear from it.
- Read $\rho$'s diagonal as the Born probabilities and its off-diagonal as the **coherence** — modulus *and* phase.
- Rewrite the Born rule, the expectation value and a gate in $\rho$'s language: $P(x) = \langle x|\rho|x\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, $\rho \to U\rho U^\dagger$ — and *earn* $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$.
- State and test the three conditions a matrix must satisfy to be a density matrix, and say what a violation means physically.

`01/11` ended on a complaint a vector cannot answer: a real device holds *"probably $|0\rangle$, with some probability something else"*, and $a_0|0\rangle + a_1|1\rangle$ has no slot for "with some probability". Today we build the object that has that slot — starting, as always, from a state you already know perfectly. The mixtures themselves arrive in `01/13`; today is about understanding the new container so well that the mixtures will feel inevitable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum import density
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state, born_probabilities, bloch_vector
from qot_course.quantum.gates import PAULI_X, PAULI_Y, PAULI_Z, HADAMARD, expectation, apply_gate

rng = np.random.default_rng(0)   # for the exercises
viz.use_course_style()

## 1. The outer product, by hand

`01/05` gave you the **bra** $\langle\psi|$ as the conjugate-transposed row of the ket $|\psi\rangle$, and the inner product $\langle\varphi|\psi\rangle$ as "row times column" — a number. Put them in the *other* order, "column times row", and you get a matrix: the **outer product**

$$ \rho = |\psi\rangle\langle\psi| . $$

For a qubit $|\psi\rangle = a_0|0\rangle + a_1|1\rangle$, write it out:

$$ \rho = \begin{pmatrix} a_0 \\ a_1 \end{pmatrix}\begin{pmatrix} \bar a_0 & \bar a_1 \end{pmatrix} = \begin{pmatrix} |a_0|^2 & a_0\bar a_1 \\ a_1\bar a_0 & |a_1|^2 \end{pmatrix} . $$

Three things you can read off before touching any code. The **diagonal** is $|a_0|^2, |a_1|^2$ — the Born probabilities of `01/04`. The **off-diagonal** entries are $a_0\bar a_1$ and its conjugate: they carry the *product* of the two amplitudes, which a $Z$ measurement never sees. And the whole matrix is built from *products of an amplitude and a conjugate amplitude* — so if you multiply $|\psi\rangle$ by a global phase $e^{i\alpha}$, every entry picks up $e^{i\alpha}\,\overline{e^{i\alpha}} = 1$: **$\rho$ does not change.** The unphysical freedom of `01/03` is gone; $\rho$ is the state with nothing extra.

Two examples by hand: for $|+\rangle$, $a_0 = a_1 = 1/\sqrt2$, so every entry is $\tfrac12$; for $|0\rangle$, $a_0 = 1, a_1 = 0$, so $\rho = \mathrm{diag}(1, 0)$. Let's check both, then look at a tilted state with a phase, where the off-diagonal becomes complex.

In [ ]:
def outer(psi):
    """|psi><psi| by hand: column times conjugate row."""
    return np.outer(psi, psi.conj())

rho_plus = outer(KET_PLUS)
rho_zero = outer(KET_0)
print("rho for |+> =\n", np.round(rho_plus, 3))
print("rho for |0> =\n", np.round(rho_zero, 3))
print("same as the library's density.density_matrix:", np.allclose(rho_plus, density.density_matrix(KET_PLUS)), np.allclose(rho_zero, density.density_matrix(KET_0)))

# a tilted state with a phase: theta = pi/3, phi = pi/5 (the state of 01/10 and 01/11)
theta_t, phi_t = np.pi / 3, np.pi / 5
psi = qubit_state(theta=theta_t, phi=phi_t)
rho = outer(psi)
print("\ntilted state psi =", np.round(psi, 4))
print("rho =\n", np.round(rho, 4))

# a global phase leaves rho untouched
psi_phased = np.exp(1j * 0.8) * psi
print("\n|<psi_phased|psi>| =", round(abs(np.vdot(psi_phased, psi)), 6), " (same ray, 01/03);  rho identical:", np.allclose(outer(psi_phased), rho))

viz.plot_density_matrix(rho, title="rho of the tilted state  (theta = pi/3, phi = pi/5)", basis_labels=["|0>", "|1>"])
plt.show()

**Read the output and the figure.** The hand products agree with the library to the digit: all $\tfrac12$ for $|+\rangle$, $\mathrm{diag}(1,0)$ for $|0\rangle$. For the tilted state, the diagonal reads $(0.75, 0.25)$ — exactly $\cos^2(\pi/6), \sin^2(\pi/6)$, the Born shadow of `01/04` — and the off-diagonal is *complex*: the heatmaps show a non-zero imaginary part, antisymmetric across the diagonal (because $\rho_{10} = \bar\rho_{01}$ — $\rho$ is Hermitian). Multiplying $|\psi\rangle$ by $e^{0.8i}$ moves the vector but leaves $\rho$ identical to machine precision. Two vectors that differ by a global phase are one physical state, and $\rho$ says so automatically. Be clear about the honest limit, though: for a *pure* state $\rho$ holds nothing $|\psi\rangle$ did not — it only forgets the global phase, which is exactly right. Its extra capacity is spent only in `01/13`.

## 2. Where the information lives: diagonal and off-diagonal

Use the Bloch parametrisation of `01/03`, $|\psi\rangle = \cos\tfrac\theta2\,|0\rangle + e^{i\phi}\sin\tfrac\theta2\,|1\rangle$, and read the entries of $\rho$ in those terms:

$$ \rho_{00} = \cos^2\tfrac\theta2 = \tfrac{1 + \cos\theta}{2}, \qquad \rho_{11} = \sin^2\tfrac\theta2 = \tfrac{1 - \cos\theta}{2}, \qquad \rho_{01} = \cos\tfrac\theta2\sin\tfrac\theta2\, e^{-i\phi} = \tfrac12\sin\theta\; e^{-i\phi} . $$

So the **diagonal** knows only $\theta$ — the latitude on the sphere, which is all a $Z$ measurement can see (`01/04`). The **off-diagonal** entry $\rho_{01}$ — the **coherence** — carries two things: its *modulus* $\tfrac12\sin\theta$, largest on the equator (equal superpositions) and zero at the poles (the classical bit), and its *phase* $-\phi$, the longitude — the relative phase that `01/02` showed is physical and that $Z$ is blind to. Nothing about the state is lost: $(\theta, \phi)$ can be read back from $\rho$, and the global phase is the only thing that cannot, because it was never physical.

Let's walk down a meridian ($\phi = \pi/5$ fixed, $\theta$ from $0$ to $\pi$) and watch the three numbers move.

In [ ]:
thetas = np.linspace(0, np.pi, 200)
rhos = [outer(qubit_state(theta=t, phi=phi_t)) for t in thetas]
rho00 = np.array([r[0, 0].real for r in rhos])
coh_mod = np.array([abs(r[0, 1]) for r in rhos])
coh_arg = np.array([np.angle(r[0, 1]) for r in rhos])

print("checks on the tilted state (theta = pi/3):")
print("  rho_00 = (1 + cos theta)/2 :", round(rho[0, 0].real, 4), "=", round((1 + np.cos(theta_t)) / 2, 4))
print("  |rho_01| = sin(theta)/2    :", round(abs(rho[0, 1]), 4), "=", round(np.sin(theta_t) / 2, 4))
print("  arg rho_01 = -phi          :", round(np.angle(rho[0, 1]), 4), "=", round(-phi_t, 4))

phis = np.linspace(0, 2 * np.pi, 200)                  # now walk a parallel: theta fixed at pi/3, phi around the circle
rhos_par = [outer(qubit_state(theta=theta_t, phi=f)) for f in phis]
par_mod = np.array([abs(r[0, 1]) for r in rhos_par])
par_arg = np.unwrap([np.angle(r[0, 1]) for r in rhos_par])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
ax.plot(thetas, rho00, color=COLORS["quantum"], lw=2, label=r"$\rho_{00}$ = P(0), the diagonal")
ax.plot(thetas, coh_mod, color=COLORS["highlight"], lw=2, label=r"$|\rho_{01}|$, the coherence")
ax.axvline(theta_t, color=COLORS["grid"], lw=1.5, ls="--")
ax.text(theta_t + 0.05, 0.9, "tilted state", fontsize=9)
ax.set_xticks([0, np.pi / 2, np.pi]); ax.set_xticklabels([r"$0$ (|0>)", r"$\pi/2$ (equator)", r"$\pi$ (|1>)"])
ax.set_xlabel(r"polar angle $\theta$ (down a meridian, $\phi = \pi/5$)"); ax.set_ylim(0, 1.05)
ax.set_title("Down a meridian: diagonal = latitude,\ncoherence peaks at the equator", pad=12)
ax.legend(fontsize=8)
ax = axes[1]
ax.plot(phis, par_mod, color=COLORS["highlight"], lw=2, label=r"$|\rho_{01}|$ (constant: $\sin\theta/2$)")
ax.plot(phis, par_arg / np.pi, color=COLORS["flow"], lw=2, label=r"$\arg\rho_{01}/\pi$ (= $-\phi/\pi$)")
ax.plot(phis, rho00[np.argmin(abs(thetas - theta_t))] * np.ones_like(phis), color=COLORS["quantum"], lw=2, ls=":", label=r"$\rho_{00}$ (constant: $Z$ is blind to $\phi$)")
ax.set_xticks([0, np.pi, 2 * np.pi]); ax.set_xticklabels([r"$0$", r"$\pi$", r"$2\pi$"])
ax.set_xlabel(r"azimuth $\phi$ (around a parallel, $\theta = \pi/3$)"); ax.set_ylabel("value (see legend)")
ax.set_title("Around a parallel: only the phase\nof the coherence moves", pad=12)
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

**Read the output and the figures.** The three closed forms hold to four digits on the tilted state. Left: walking from the north pole to the south pole, the diagonal entry $\rho_{00}$ falls from $1$ to $0$ as $(1 + \cos\theta)/2$ while the coherence $|\rho_{01}|$ rises from $0$ to its maximum $\tfrac12$ on the equator and falls back to $0$ — the poles are the classical bit (no superposition, no coherence), the equator is the most quantum the qubit gets. Right: walking *around* a parallel at fixed $\theta$, the diagonal and the modulus of the coherence do not move at all — $Z$ is blind to the longitude, and so is the *size* of the superposition — while the phase of the coherence runs linearly as $-\phi$: it *is* the longitude. When `01/13` puts a state beside $|+\rangle$ with the same diagonal and a zero off-diagonal, you will know exactly what has been lost — the coherence — and that a $Z$ measurement could never have told you.

## 3. Everything you know, rewritten in $\rho$'s language

$\rho$ would be a curiosity if it only restated $|\psi\rangle$. Its power is that every prediction you learned to make from a vector can be made from $\rho$ *by a trace* — and traces are linear, which is what will let `01/13` average over ignorance. Three rewrites, each by hand.

**The Born rule.** $P(x) = |\langle x|\psi\rangle|^2 = \langle x|\psi\rangle\langle\psi|x\rangle = \langle x|\rho|x\rangle$ — the diagonal entry, as section 1 said. In any other basis $\{|b_k\rangle\}$ (`01/08`), the same line gives $P(k) = \langle b_k|\rho|b_k\rangle$.

**The expectation value.** The bracket of `01/07` and `01/10`, $\langle A\rangle = \langle\psi|A|\psi\rangle$, becomes a trace. Write the trace of the product $\rho A$ with components, $\mathrm{tr}(\rho A) = \sum_{x}(\rho A)_{xx} = \sum_{x,y}\rho_{xy}A_{yx} = \sum_{x,y} a_x\bar a_y A_{yx} = \sum_{y}\bar a_y\big(\sum_x A_{yx}a_x\big) = \langle\psi|A|\psi\rangle$. So

$$ \langle A\rangle = \mathrm{tr}(\rho A) . $$

**A gate.** If $|\psi\rangle \to U|\psi\rangle$, then $\rho = |\psi\rangle\langle\psi| \to U|\psi\rangle\langle\psi|U^\dagger$, because the bra of $U|\psi\rangle$ is $\langle\psi|U^\dagger$ (`01/06`): $\rho \to U\rho U^\dagger$.

And now the formula `01/10` asked you to accept. The four matrices $I, X, Y, Z$ are a basis of all $2\times2$ matrices, and they are orthogonal under the trace: $\mathrm{tr}(\sigma_i\sigma_j) = 2\delta_{ij}$, $\mathrm{tr}(\sigma_i) = 0$ (`01/07`: the Paulis square to $I$, and products of two different Paulis are $\pm i$ times the third, traceless). So any $\rho$ expands as $\rho = c_0 I + c_x X + c_y Y + c_z Z$, with coefficients read off by traces: $c_0 = \mathrm{tr}(\rho)/2 = \tfrac12$ and $c_i = \mathrm{tr}(\rho\,\sigma_i)/2 = \langle\sigma_i\rangle/2$ — the Bloch coordinate, by the rewrite we made a moment ago. Hence

$$ \rho = \tfrac12\big(I + \langle X\rangle X + \langle Y\rangle Y + \langle Z\rangle Z\big) = \tfrac12\big(I + \vec r\cdot\vec\sigma\big), $$

earned, not previewed. Written out, $\rho = \tfrac12\begin{pmatrix} 1 + z & x - iy \\ x + iy & 1 - z\end{pmatrix}$: the diagonal is $(1 \pm z)/2$ — the $Z$ probabilities of `01/10` — and the coherence is $(x - iy)/2$, so its real part is half of $\langle X\rangle$ and its imaginary part is minus half of $\langle Y\rangle$: "the phase lives off-diagonal" now has coordinates. The density matrix and the Bloch vector are the same information, packed two ways — and the three shot-estimated averages of `01/10` §3 *are* a measurement of $\rho$. Let's check all of it on the tilted state.

In [ ]:
# Born rule as a diagonal entry, in Z and in X
P = born_probabilities(psi)
print("Born P(0): |<0|psi>|^2 =", round(P["0"], 4), "   <0|rho|0> =", round(rho[0, 0].real, 4))
print("Born P(+): |<+|psi>|^2 =", round(abs(np.vdot(KET_PLUS, psi)) ** 2, 4), "   <+|rho|+> =", round(np.vdot(KET_PLUS, rho @ KET_PLUS).real, 4))

# expectation value as a trace: three Paulis and a non-Pauli observable
A = (PAULI_X + PAULI_Z) / np.sqrt(2)
for name, M in [("X", PAULI_X), ("Y", PAULI_Y), ("Z", PAULI_Z), ("(X+Z)/sqrt2", A)]:
    print(f"<{name}>: bracket <psi|A|psi> = {expectation(psi, M):+.4f}   tr(rho A) = {np.trace(rho @ M).real:+.4f}")

# the Pauli expansion: rho = (I + rx X + ry Y + rz Z)/2, coefficients read off by traces
r = np.array([np.trace(rho @ M).real for M in (PAULI_X, PAULI_Y, PAULI_Z)])
rho_from_r = 0.5 * (np.eye(2) + r[0] * PAULI_X + r[1] * PAULI_Y + r[2] * PAULI_Z)
print("\nBloch vector from traces tr(rho sigma_i):", np.round(r, 4), "   from the amplitudes (01/03):", np.round(bloch_vector(psi), 4))
print("(I + r.sigma)/2 == rho :", np.allclose(rho_from_r, rho), "   library density_from_bloch / bloch_vector agree:", np.allclose(density.density_from_bloch(r), rho), np.allclose(density.bloch_vector(rho), r))
print("tr(sigma_i sigma_j) = 2 delta_ij:", np.round([[np.trace(Mi @ Mj).real for Mj in (PAULI_X, PAULI_Y, PAULI_Z)] for Mi in (PAULI_X, PAULI_Y, PAULI_Z)], 4).tolist())

# a gate acts as rho -> U rho U^dagger, and the Bloch vector rotates rigidly (01/06)
rho_after = HADAMARD @ rho @ HADAMARD.conj().T
print("\nH: U rho U^dag == |H psi><H psi| :", np.allclose(rho_after, outer(apply_gate(HADAMARD, psi))),
      "  Bloch vector before", np.round(r, 3), "-> after", np.round(density.bloch_vector(rho_after), 3), " (x <-> z, y -> -y: the H half-turn of 01/08)")

**Read the output.** Every bracket equals the corresponding trace, for the three Paulis and for an observable that is not one. The Bloch vector read off by traces is the one `01/03` computed from the amplitudes, and $\tfrac12(I + \vec r\cdot\vec\sigma)$ rebuilds $\rho$ exactly — the library's `density_from_bloch` and `bloch_vector` are these two lines. The Pauli orthogonality table is $2I$, as claimed. And $H\rho H^\dagger$ is the density matrix of $H|\psi\rangle$, with the Bloch vector swapped $x \leftrightarrow z$ and $y \to -y$: the half-turn of `01/08`, now as a conjugation of matrices. You have not lost a single prediction by trading the vector for the matrix — and the matrix forgets only the global phase.

## 4. What makes a matrix a density matrix

Turn it around: given a $2\times2$ matrix, when is it *some* state's $\rho$? Three conditions, each with a physical meaning.

1. **Hermitian**, $\rho = \rho^\dagger$: the diagonal is real (probabilities), and $\rho_{10} = \bar\rho_{01}$ (one coherence, not two).
2. **Unit trace**, $\mathrm{tr}\rho = 1$: the Born probabilities of *any* basis sum to one, since $\sum_k\langle b_k|\rho|b_k\rangle = \mathrm{tr}\rho$.
3. **Positive**, $\langle\varphi|\rho|\varphi\rangle \ge 0$ for every vector $|\varphi\rangle$ — equivalently, no negative eigenvalue (a Hermitian matrix has real eigenvalues and an orthonormal eigenbasis, `01/07`, so expanding $|\varphi\rangle$ in it gives $\langle\varphi|\rho|\varphi\rangle = \sum_k \lambda_k|\langle e_k|\varphi\rangle|^2$, non-negative for all $|\varphi\rangle$ exactly when every $\lambda_k \ge 0$): the probability $\langle b|\rho|b\rangle$ of *every* possible outcome in *every* possible basis is non-negative.

For a pure state all three are immediate by hand: $\rho^\dagger = (|\psi\rangle\langle\psi|)^\dagger = |\psi\rangle\langle\psi|$; $\mathrm{tr}\rho = |a_0|^2 + |a_1|^2 = 1$; and $\langle\varphi|\rho|\varphi\rangle = |\langle\varphi|\psi\rangle|^2 \ge 0$. More is true for a pure state: $\rho^2 = |\psi\rangle\langle\psi|\psi\rangle\langle\psi| = \rho$ — it is a **projector** onto the line of $|\psi\rangle$, with eigenvalues $1$ (on $|\psi\rangle$) and $0$ (on the orthogonal state). Keep that in your pocket: `01/13` will measure how far a state is from being a projector — $\mathrm{tr}(\rho^2)$, which equals $1$ exactly when $\rho^2 = \rho$ — and call it purity.

Let's test the three conditions on the states above and on a tempting impostor — Hermitian, unit trace, and yet not a state.

In [ ]:
def report(name, M):
    evals = np.linalg.eigvalsh(M)
    print(f"{name:28s} Hermitian: {np.allclose(M, M.conj().T)!s:5}  trace: {np.trace(M).real:.3f}  eigenvalues: {np.round(evals, 3)}  -> valid: {density.is_density_matrix(M)}")

report("rho of |+>", rho_plus)
report("rho of |0>", rho_zero)
report("rho of the tilted state", rho)
print("pure => projector: rho @ rho == rho :", np.allclose(rho @ rho, rho), "  eigenvalues (1, 0): eigenvector of 1 is psi up to phase:", round(abs(np.vdot(np.linalg.eigh(rho)[1][:, 1], psi)), 6))

impostor = np.array([[0.5, 0.9], [0.9, 0.5]])
report("impostor [[.5,.9],[.9,.5]]", impostor)
# where does the negative number show up? in the X basis: <-|impostor|-> = 0.5 - 0.9
p_minus = np.vdot(KET_MINUS, impostor @ KET_MINUS).real
print(f"impostor's 'probability' of the outcome |-> in the X basis: <-|M|-> = {p_minus:+.3f}   (a negative probability -- no state can do this)")
# the classical corner: a probability vector p, placed on the diagonal, IS a valid density matrix
p_classical = np.array([0.7, 0.3])
rho_classical = np.diag(p_classical).astype(complex)
report("diag(p) for p = (0.7, 0.3)", rho_classical)
print("   diagonal rho read in the Z basis gives back p:", np.round(np.diag(rho_classical).real, 3), "  coherence:", abs(rho_classical[0, 1]), "  projector (rho@rho == rho)?", np.allclose(rho_classical @ rho_classical, rho_classical))
print("largest coherence a valid rho can carry, |rho_01| <= sqrt(rho_00 rho_11):  with diagonal (0.7, 0.3):", round(np.sqrt(0.7 * 0.3), 3), ";  with diagonal (0.5, 0.5):", round(np.sqrt(0.5 * 0.5), 3), "(reached by |+>, and exceeded by the impostor's 0.9)")

**Read the output.** The three genuine density matrices pass all three tests and have eigenvalues $(1, 0)$ — projectors; the eigenvector of eigenvalue $1$ is $|\psi\rangle$ itself (overlap of modulus $1$). The impostor is Hermitian with unit trace, yet one eigenvalue is $-0.4$, so it fails positivity — and you can point to the measurement that exposes it: in the $X$ basis of `01/08`, its "probability" of $|-\rangle$ would be $0.5 - 0.9 = -0.4$. A coherence cannot be arbitrarily large: with diagonal $(\tfrac12, \tfrac12)$ it is capped at $\tfrac12$, which $|+\rangle$ reaches (in general $|\rho_{01}|^2 \le \rho_{00}\rho_{11}$ — exercise 3 proves it). One more matrix passed all three tests: $\mathrm{diag}(0.7, 0.3)$, a classical probability vector placed on the diagonal, with zero coherence. It is a valid density matrix — and *not* a projector, so it is no pure state's $|\psi\rangle\langle\psi|$ (section 5 says what it is). This is the dictionary row you are earning today: a classical $p$ *is* a diagonal $\rho$, and what $\rho$ adds to $p$ is exactly the coherence. The three conditions are not bookkeeping: they are exactly what makes every probability of every measurement a probability.

## 5. The slot a vector does not have

Here is the promise of `01/11`, kept in one paragraph and built in the next notebook. Suppose a device gives you $|0\rangle$ with probability $1 - \epsilon$ and $|1\rangle$ with probability $\epsilon$ — honest ignorance, not superposition. Can a vector describe that? Try adding: $\sqrt{1-\epsilon}\,|0\rangle + \sqrt{\epsilon}\,|1\rangle$ is a *superposition*, with a coherence $\sqrt{\epsilon(1-\epsilon)}$ that an $X$ measurement would detect as interference (`01/02`) — but an ignorant coin shows no interference at all. Vectors add amplitudes; ignorance adds *probabilities*. Now look at what section 3 bought you: every prediction is $\mathrm{tr}(\rho A)$, **linear in $\rho$**. So the average prediction over a coin that hands you $\rho_0$ or $\rho_1$ is $\mathrm{tr}\big(\,[(1-\epsilon)\rho_0 + \epsilon\rho_1]\,A\big)$ — the prediction of a single matrix

$$ \rho = (1 - \epsilon)\,|0\rangle\langle 0| + \epsilon\,|1\rangle\langle 1| = \begin{pmatrix} 1-\epsilon & 0 \\ 0 & \epsilon\end{pmatrix}, $$

Hermitian, unit trace, positive, *with zero coherence* — a valid density matrix that is **not** $|\psi\rangle\langle\psi|$ for any $|\psi\rangle$ (check: it is not a projector, $\rho^2 \ne \rho$). That is the slot. `01/13` builds such states, measures how mixed they are, and sets the most mixed one beside $|+\rangle$ — same diagonal, no coherence — the seed of the whole course.

## Your turn

1. **The classical corner.** Build $\rho$ for $|0\rangle$ and $|1\rangle$ by hand with `outer`. What are their diagonals and coherences? Compute the $X$-basis probabilities $\langle\pm|\rho|\pm\rangle$ for $|0\rangle$ — the same $\tfrac12,\tfrac12$ as `01/08` found by rotate-then-measure? Why are the poles the most "classical" qubit states?
2. **Find the phase.** Build $\rho$ for `qubit_state(np.pi/2, np.pi/2)` ($= |{+i}\rangle$ of `01/02`) and print its real and imaginary parts. Check $\arg\rho_{01} = -\phi$ and $|\rho_{01}| = \tfrac12$. Multiply the state by $e^{i\alpha}$ for three values of $\alpha$ drawn with `rng` and confirm $\rho$ does not move; then change $\phi$ instead and watch which entries move.
3. **Break a rule, then bound it.** Confirm that the impostor `[[0.5, 0.9], [0.9, 0.5]]` fails `density.is_density_matrix`, and find the eigenvector with the negative eigenvalue — which measurement would report a negative probability? Then prove by hand that any valid $2\times2$ $\rho$ satisfies $|\rho_{01}|^2 \le \rho_{00}\rho_{11}$ (hint: $\det\rho = \lambda_0\lambda_1 \ge 0$), and rescale the impostor's off-diagonal to the largest value that passes.
4. **Harder: gates on $\rho$.** Show by hand that $\rho \to U\rho U^\dagger$ keeps all three conditions of section 4 and keeps $\rho^2 = \rho$. Then, for $R_y(\theta)$ and $R_z(\phi)$ of `01/09` applied to $\rho$ of $|0\rangle$, verify numerically that `density.bloch_vector(U @ rho @ U.conj().T)` is the rotated Bloch vector (`01/06`: rigid rotation by the full angle), and that $\mathrm{tr}(\rho^2)$ is unchanged under any unitary — a first hint of why `01/13`'s purity is the right invariant.

## What you built

- You built $\rho = |\psi\rangle\langle\psi|$ by hand from two amplitudes, checked it against the library, and watched a global phase vanish from it.
- You read the diagonal as the Born probabilities (latitude) and the off-diagonal coherence as modulus $\tfrac12\sin\theta$ and phase $-\phi$ (longitude) — walking a meridian to see it.
- You rewrote the Born rule, the expectation value and a gate as $\langle b|\rho|b\rangle$, $\mathrm{tr}(\rho A)$ and $U\rho U^\dagger$, and *earned* $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ from the Pauli basis.
- You stated and tested the three conditions (Hermitian, unit trace, positive), saw a pure $\rho$ is a projector, and caught an impostor by the measurement that would give a negative probability.
- You saw why linearity in $\rho$ opens the slot for "with some probability" that no vector has.

$\rho$ is the single most important object in this course — and you built it from a state you already knew, losing nothing but a phase that was never real. Everything ahead, from mixedness to entanglement to quantum optimal transport, is written in this language.

**Dictionary so far** (the table from `01/11`; the row $p \leftrightarrow \rho$ is earned today and moves up from the forward-looking list):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| marginal | partial trace |
| transport plan | quantum coupling |

(The last two rows are still forward-looking — earned in `01/16` and `04/04`.)

## What's next

Section 5 showed the slot; `01/13_pure_vs_mixed` fills it. We build statistical mixtures $\sum_i p_i\,|\psi_i\rangle\langle\psi_i|$, meet the maximally mixed state $I/2$, learn to measure mixedness with $\mathrm{tr}(\rho^2)$ — the "how far from a projector" of section 4 — and set $I/2$ beside $|+\rangle$: same diagonal, zero coherence, different soul. That single comparison is the seed of the whole course.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §2.4 (the density operator: definition, properties, the Bloch-sphere picture for mixed states), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- L. Landau, *Das Dämpfungsproblem in der Wellenmechanik*, Z. Phys. **45**, 430–441 (1927) — the density matrix appears here and, independently and with the general trace formalism, in J. von Neumann, *Wahrscheinlichkeitstheoretischer Aufbau der Quantenmechanik*, Gött. Nachr. 245–272 (1927); book form: J. von Neumann, *Mathematische Grundlagen der Quantenmechanik*, Springer (1932). doi:10.1007/BF01343064 (Landau); doi:10.1007/978-3-642-61409-5 (von Neumann 1932, reprint)
- U. Fano, *Description of States in Quantum Mechanics by Density Matrix and Operator Techniques*, Rev. Mod. Phys. **29**, 74–93 (1957) — the classic pedagogical account, including the Pauli (multipole) expansion. doi:10.1103/RevModPhys.29.74

**Previous:** `notebooks/01_QuantumFoundations/11_shot_noise_and_first_circuit.ipynb` · **Next:** `notebooks/01_QuantumFoundations/13_pure_vs_mixed.ipynb`